# Transformers Docs Copilot — grounded RAG baseline

Answers questions about a **version-pinned** slice of the Hugging Face Transformers docs
and cites the retrieved section. This notebook runs the **untuned 1.5B baseline** that the
QLoRA adapter must beat.

- Docs: `huggingface/transformers` @ `v5.24.0` (pinned in `data/raw/docset.json`)
- Model: `Qwen/Qwen2.5-1.5B-Instruct`, 4-bit NF4 on a free Colab T4
- Evaluation: 43 held-out test questions (28 answerable, 15 must abstain)
- Full run: about 6–10 minutes after the model download. Set `QUICK_LIMIT = 2` for a
  two-question plumbing check.

**Before running:** select `Runtime → Change runtime type → T4 GPU`.

In [ ]:
# 0 means the full test split. Set to 2 (or 3) for a cheap end-to-end check.
QUICK_LIMIT = 0
SPLIT = "test"

PREFIX = "results/answers_quick" if QUICK_LIMIT else "results/answers_baseline"
LABELS = "results/label_sheet_quick.csv" if QUICK_LIMIT else "results/label_sheet_baseline.csv"
print("output:", PREFIX, "|", LABELS)

## 1. GPU gate

4-bit inference needs CUDA. If this cell fails, fix the runtime type before continuing.

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
props = torch.cuda.get_device_properties(0)
print(torch.__version__, "|", props.name, f"| {props.total_memory / 1e9:.1f} GB VRAM")

## 2. Get the project

**Option A (recommended):** this repo lives on GitHub. Paste its URL below and rerun.

- The repo is **private**, so the clone cell needs a read-only token.
- Create a fine-grained PAT: GitHub → Settings → Developer settings → Fine-grained
  tokens → Generate new token. Repository access: only `transformers-docs-copilot`;
  permission: **Contents → Read-only**; short expiry.
- In Colab, click the key icon in the left sidebar → add a secret named `GITHUB_TOKEN`,
  paste the token, and enable notebook access. The cell uses it only for the clone and
  then removes it from the git remote.
- The token is never printed; errors are scrubbed before display.

**Option B:** zip the project folder (keep the top-level folder name
`transformers-docs-copilot`), upload it to Colab, set `ZIP_PATH`, and rerun.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/titidintsako-bit/transformers-docs-copilot.git"
ZIP_PATH = ""   # e.g. "/content/transformers-docs-copilot.zip"

GITHUB_TOKEN = ""
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    pass

PROJECT_DIR = Path("/content/transformers-docs-copilot")
if not PROJECT_DIR.exists() and ZIP_PATH:
    subprocess.run(["unzip", "-q", ZIP_PATH, "-d", "/content"], check=True)
if not PROJECT_DIR.exists():
    if not REPO_URL:
        raise SystemExit("Set REPO_URL (recommended) or ZIP_PATH, then rerun this cell.")
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"
    if GITHUB_TOKEN and REPO_URL.startswith("https://github.com/"):
        env.update({
            "GIT_CONFIG_COUNT": "1",
            "GIT_CONFIG_KEY_0": "url.https://x-access-token:%s@github.com/.insteadOf" % GITHUB_TOKEN,
            "GIT_CONFIG_VALUE_0": "https://github.com/",
        })
    result = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)],
        env=env, capture_output=True, text=True,
    )
    if result.returncode != 0:
        message = result.stderr.replace(GITHUB_TOKEN, "***") if GITHUB_TOKEN else result.stderr
        raise SystemExit("git clone failed:\n" + message[-1200:])
    if GITHUB_TOKEN:
        subprocess.run(
            ["git", "-C", str(PROJECT_DIR), "remote", "set-url", "origin", REPO_URL],
            check=True,
        )
os.chdir(PROJECT_DIR)
print("project:", Path.cwd())


## 3. Install pinned dependencies

Colab ships a recent PyTorch; the `ml` and `gpu` extras pin transformers, accelerate and
bitsandbytes. If the Hub rate-limits the model download, run
`from huggingface_hub import login; login()` first (a read token is enough).

If pip asks you to restart the runtime, do so and rerun from the project cell; the install is cached.

In [ ]:
!pip install -q -e ".[ml,gpu]"
import transformers
print("transformers", transformers.__version__)

## 4. Verify the corpus and rebuild the BM25 index

The committed markdown and chunks are byte-checked against the manifest. `artifacts/` is
not committed, so the index is rebuilt here; this is offline and deterministic.

In [ ]:
!python -m src.fetch_docs --check
!python -m src.eval_questions
!python -m src.index

## 5. Run the untuned baseline

For every test question: BM25 top-5 → numbered passages → model answer → parser. The run
scores format, abstention, citation validity and gold-section citation hits, then writes:

- `results/answers_baseline.md` and `.json`
- `results/predictions_baseline.jsonl` (full outputs)
- `results/label_sheet_baseline.csv` (human columns still blank)

In [ ]:
cmd = [
    sys.executable, "-m", "src.eval_answers",
    "--split", SPLIT,
    "--model", "base",
    "--quantize", "on",
    "--out-prefix", PREFIX,
    "--labels", LABELS,
]
if QUICK_LIMIT:
    cmd += ["--limit", str(QUICK_LIMIT)]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

## 6. Inspect the report

`behavior_accuracy` is the headline contract metric. `grounded_answer_rate` is stricter:
answered **and** cited a gold section. `format_rate` being low is expected for the
untuned model; that gap is what the QLoRA stage targets.

In [ ]:
import json
from IPython.display import Markdown, display

report = Path(f"{PREFIX}.json")
payload = json.loads(report.read_text())
print(json.dumps(payload["overall"], indent=2))
print()
display(Markdown(Path(f"{PREFIX}.md").read_text()))

In [ ]:
for row in payload["rows"][:3]:
    print(f"--- {row['qid']} [{row['status']}] {row['question']}")
    print(row["answer"][:500])
    print("citations:", [(source["n"], source["url"]) for source in row["sources"]])

## 7. Checkpoint the artifacts to Drive

Safe to skip, but Colab disconnects. The label sheet is the hand-scoring input; fill the
blank rubric columns after following `docs/EVAL.md`.

In [ ]:
import shutil
from google.colab import drive

drive.mount("/content/drive")
dest = Path("/content/drive/MyDrive/transformers-docs-copilot-results")
dest.mkdir(parents=True, exist_ok=True)

margin = Path(PREFIX).stem.removeprefix("answers_")
predictions = Path(PREFIX).parent / f"predictions_{margin}.jsonl"
for path in [Path(f"{PREFIX}.md"), Path(f"{PREFIX}.json"), Path(LABELS), predictions]:
    if path.exists():
        shutil.copy2(path, dest / path.name)
        print("saved", dest / path.name)

## 8. What to do with the numbers

- Commit the generated `results/answers_baseline.md`/`.json` (the JSONL stays local).
- Fill the label sheet: `supported_0_1_2`, `citation_precision_0_1`, `citation_recall_0_1`.
- Keep this split untouched for tuning: no test-page passages may enter fine-tuning data.
- Next milestone: QLoRA the same base model on dev data only, then re-run this notebook
  with the adapter and compare the same table.